In [3]:
import os
import numpy as np

from hakken_models.datasets.deployment import DatasetDeployment


/home/pablo.sanchez2/GitHub/project_spaice_ds/packages/pip/hakken-models/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
root_folder = "s3://sai-spaice-ds/data/processed/data_processing/zenml/pubtator3-v0.4.0/v2"

dataset = DatasetDeployment(root_folder)


2026-03-09 11:33:58.189 | INFO     | hakken_models.datasets.deployment:__init__:77 - Loading dataset artifacts for s3://sai-spaice-ds/data/processed/data_processing/zenml/pubtator3-v0.4.0/v2
2026-03-09 11:33:59.367 | INFO     | hakken_models.datasets.deployment:_load_npy:266 - Loading tensor: s3://sai-spaice-ds/data/processed/data_processing/zenml/pubtator3-v0.4.0/v2/tensors/train.npy
2026-03-09 11:34:03.820 | INFO     | hakken_models.datasets.deployment:_load_npy:266 - Loading tensor: s3://sai-spaice-ds/data/processed/data_processing/zenml/pubtator3-v0.4.0/v2/tensors/val.npy
2026-03-09 11:34:04.410 | INFO     | hakken_models.datasets.deployment:_load_npy:266 - Loading tensor: s3://sai-spaice-ds/data/processed/data_processing/zenml/pubtator3-v0.4.0/v2/tensors/test.npy
2026-03-09 11:34:04.448 | SUCCESS  | hakken_models.datasets.deployment:__init__:111 - Mappings and tensors successfully loaded.
2026-03-09 11:34:04.448 | INFO     | hakken_models.datasets.deployment:_load_npy:266 - Loadin

In [6]:
dataset._node_embeddings_np.shape

(580432, 1024)

# Latent Explanations

In [ ]:
import os
from pathlib import Path
import polars as pl
from hakken_explainer.candidate_finder.latent.llm import LLMCandidateFinder
import json

In [2]:
data_folder = Path("/home/pablo.sanchez2/data/pubtator3-v0.4.0")

facts_file = data_folder / "edges.tsv"
nodes_file = data_folder / "nodes_corrected.tsv"
node_names = data_folder / "node_id_to_names_mapping.jsonl"


facts_pl = pl.read_csv(facts_file, separator="\t")
nodes_pl = pl.read_csv(nodes_file, separator="\t")

In [3]:
facts_pl.head()

subject_id,relation_type,object_id,year,year_occurrences,subject_domain,object_domain,subject_id_raw,object_id_raw,number_of_occurrences
str,str,str,i64,str,str,str,str,str,i64
"""0003fb22e2049da4ab2c993efa06f7…","""ASSOCIATE""","""2a57ef5e287a4d2b1bc3cb68a39bf1…",2014,"""2014""","""GENE""","""GENE""","""12400""","""12842""",1
"""0003fb22e2049da4ab2c993efa06f7…","""ASSOCIATE""","""87a39259584f0df9790286fa4ce057…",1994,"""1994|2015""","""GENE""","""GENE""","""12400""","""17523""",2
"""0003fb22e2049da4ab2c993efa06f7…","""NEGATIVE_CORRELATE""","""f229d8118bdbd64429226bdc28f03f…",2018,"""2018""","""GENE""","""GENE""","""12400""","""54199""",1
"""0004388cac249917f776baf7bcd447…","""PREVENT""","""38720b76d83a57a04d5b763dc08634…",2009,"""2009""","""DNAMUTATION""","""DISEASE""","""HGVS:c.1527_1528del;Correspond…","""MESH:D006509""",1
"""00048b289b2a5de15aa7833e5fc934…","""NEGATIVE_CORRELATE""","""a4a1037a390964cd8ae630808fec7b…",2017,"""2017""","""GENE""","""GENE""","""14313""","""16323""",1


In [4]:
nodes_pl.head()

node_id,node_domain,node_id_raw,node_name,node_domain_id
str,str,str,str,str
"""0009ad04c8987619d66a00dceb4386…","""GENE""","""29547397""","""PBOR_RS12800""","""1fb687612b24dddaac8cb9635b9462…"
"""001c6cfebaba7d7927b0a197195fb7…","""DNAMUTATION""","""RS#:4950928;CorrespondingGene:…","""RS#:4950928; (CHI3L1)""","""6ef835a9c3c62d595f324e50781265…"
"""00263ee4fbf9e985397b408af40fb1…","""DNAMUTATION""","""HGVS:c.110-2A>T;CorrespondingG…","""HGVS:c.110-2A>T; (ACAD8)""","""6ef835a9c3c62d595f324e50781265…"
"""0033a95d509b8116c1318d23fd986a…","""DNAMUTATION""","""HGVS:c.14C&gt;T""","""HGVS:c.14C&gt;T""","""6ef835a9c3c62d595f324e50781265…"
"""003b8558ea81c0890ecdefeed92c27…","""DNAMUTATION""","""RS#:199422241(Expired);HGVS:c.…","""RS#:199422241(Expired);HGVS:c.…","""6ef835a9c3c62d595f324e50781265…"


In [5]:
model = "alibayram/medgemma:27b"
model = "llama3.1:8b"
# model = "llama3.1:70b"
model = "myaniu/qwen2.5-1m:7b"
cand_finder = LLMCandidateFinder(
    nodes_df=nodes_pl.to_pandas(),
    facts_df=facts_pl.to_pandas(),
    kg=None,
    model=model
)

a = """
Describe this chemical in a short paragraph "2,4-dioxo-N-[4-(4-pyridyl)phenyl]-1H-quinazoline-6-sulfonamide"
"""

In [6]:
entities = nodes_pl["node_name"].unique()
relations = facts_pl["relation_type"].unique()

chemical_entities = nodes_pl.filter(pl.col("node_domain") == "CHEMICAL")["node_name"].unique() 

disese_entities = nodes_pl.filter(pl.col("node_domain") == "DISEASE")["node_name"].unique()


In [7]:
subject = chemical_entities.sample(1)[0]
subject_domain = "CHEMICAL"
relation = "TREAT"
target = disese_entities.sample(1).item()
object_domain = "DISEASE"

print(f"[{subject}] -- [{relation}] --> [{target}]")

[methyl sterculate] -- [TREAT] --> [extra-abdominal and intra-abdominal (mainly mesenteric) fibromatosis]


In [8]:
prompt = cand_finder.build_template_prompt(
            subject, subject_domain, relation, target, object_domain
        )

# print(prompt)

In [ ]:
# from hakken_explainer.candidate_finder.entities import TemplatePathwaySet

# prompt = cand_finder.build_template_prompt(subject,  subject_domain, relation, target, object_domain)


# response = cand_finder.client.generate(model=
# "llama3.1:8b", prompt=prompt, format="json")
# result_json = json.loads(response["response"])
# template_pathway_set =  TemplatePathwaySet(**result_json)


In [10]:
template_pathways = cand_finder.find_template_pathways(subject,  subject_domain, relation, target, object_domain)


In [11]:
for pathway in template_pathways:
    print(pathway)
    print("---------------")

1. (CHEMICAL) --[ASSOCIATE]--> (PROTEINMUTATION)
2. (PROTEINMUTATION) --[CAUSE]--> (DISEASE)
---------------
1. (CHEMICAL) --[NEGATIVE_CORRELATE]--> (SNP)
2. (SNP) --[ASSOCIATE]--> (DISEASE)
---------------
1. (CHEMICAL) --[TREAT]--> (GENE)
2. (GENE) --[INHIBIT]--> (DISEASE)
---------------


In [12]:
pathways = cand_finder.instantiate_templates(subject,  subject_domain, relation, target, object_domain, template_pathways[:1])


In [13]:
for pathway in pathways:
    print(pathway)

(METHYL STERCULATE:CHEMICAL) --[ASSOCIATE]--> (TP53:PROTEIN) → (TP53:PROTEIN) --[CAUSE]--> (EXTRA-ABDOMINAL AND INTRA-ABDOMINAL FIBROMATOSIS:DISEASE)


# Embeddings

In [ ]:
from hakken_explainer.vectorstore import Vectorstore
from langchain_community.retrievers import BM25Retriever
from langchain_community.retrievers import BM25Retriever, EnsembleRetriever

In [17]:
vectorstore = Vectorstore()


In [20]:
vectorstore.add_documents_from_dataframe(
    df=cand_finder.nodes_df.iloc[:1000], 
    content_col="node_name",
    metadata_cols=["node_id", "node_domain"],
    dedup_field = "node_id",
    batch_size = 128,
    doc_type="entity")

2025-10-28 09:25:03.888 | INFO     | simple_xkgc.vectorstore.engine:add_documents_from_dataframe:145 - Initial DataFrame size: 1000
2025-10-28 09:25:03.890 | INFO     | simple_xkgc.vectorstore.engine:add_documents_from_dataframe:162 - Found 10 existing documents in vectorstore
2025-10-28 09:25:03.891 | INFO     | simple_xkgc.vectorstore.engine:add_documents_from_dataframe:168 - Filtered 10 documents already in vectorstore
2025-10-28 09:25:03.892 | INFO     | simple_xkgc.vectorstore.engine:add_documents_from_dataframe:178 - Processing 990 new documents
2025-10-28 09:25:05.422 | DEBUG    | simple_xkgc.vectorstore.engine:add_documents_from_dataframe:197 - Processed batch: 128 documents (Total added: 128)
2025-10-28 09:25:06.859 | DEBUG    | simple_xkgc.vectorstore.engine:add_documents_from_dataframe:197 - Processed batch: 128 documents (Total added: 256)
2025-10-28 09:25:08.321 | DEBUG    | simple_xkgc.vectorstore.engine:add_documents_from_dataframe:197 - Processed batch: 128 documents (T

990

In [22]:
small_df = cand_finder.nodes_df.iloc[:1000]
entity = small_df.sample(1).iloc[0]
entity

node_id                            220799a754dbacd5f24290e5839ea338
node_domain                                             DNAMUTATION
node_id_raw       RS#:121912978;HGVS:g.7757C>T;CorrespondingGene...
node_name                   RS#:121912978;HGVS:g.7757C>T; (CYP11B2)
node_domain_id                     6ef835a9c3c62d595f324e5078126567
Name: 751, dtype: object

In [29]:
await vectorstore.vectorstore.asearch("RS#:121912978;HGVS:g.7757C>T", search_type="mmr", k=10,
    filter={"keywords": "#:121912978;H"})


[]

In [ ]:
await vectorstore.vectorstore.asimilarity_search_with_relevance_scores("GLCXN2", filter={"node_domain": "GENE"})

# Node Mapping

In [ ]:
df = pl.read_ndjson(node_names)
df_exploded = df.explode("node_names")
df_normalized = df_exploded.unnest("node_names")

df_filtered = df_normalized.filter(pl.col("count") > 0)


In [ ]:
node_id = nodes_pl[10]["node_id"].item()
node_id
nodes_pl.filter(pl.col("node_id") == node_id)

In [ ]:
df_filtered.filter(pl.col("node_id") == node_id)

# MultiDiGraph

In [ ]:
import networkx as nx
import torch
import time 
from pathlib import Path

from hakken_ml_toolkit.ml_base_structures import KnowledgeGraph

In [ ]:
kg_path = Path("/home/pablo.sanchez2/Documents/GitHub/data/hakken_bio/pubtator3-v0.4.0/cached/v1.1.0")
kg = KnowledgeGraph.load(kg_path)

# Example facts_batch representing a knowledge graph
# Each row is [subject_id, relation_id, object_id]
facts_batch = kg.facts_dict["train"]

In [ ]:
facts_batch = torch.tensor([
    [0, 1, 1],
    [1, 1, 2],
    [0, 2, 1]
])

In [ ]:

G = nx.MultiDiGraph()

# Convert to list for faster iteration
facts_list = facts_batch.tolist()



tic = time.time()

# Add edges with relation as edge attribute
for subject, relation, obj in facts_list:
    G.add_edge(subject, obj, relation=relation)

delay = time.time() - tic
print(f"Time2: {delay:.2f}")

In [ ]:
G_und = G.to_undirected()

In [ ]:
for edge in G.edges:
    print(edge)

In [ ]:
for edge in G_und.edges:
    print(edge)

In [ ]:
print(G.get_edge_data(0, 1))
print(G_und.get_edge_data(1, 0))


In [ ]:
G.get_edge_data(0, 1, 0)

In [ ]:
a = [0, 1]
b = [1]


In [ ]:
list(nx.all_shortest_paths(G.to_undirected(),2, 0))

In [ ]:

# Print graph information
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"\nNodes: {list(G.nodes())}")

# Print all edges with their relations
print("\nEdges:")
for u, v, key, data in G.edges(keys=True, data=True):
    print(f"  {u} --[relation {data['relation']}]--> {v}")

# Example: Find all edges from node 0 (Alice)
print(f"\nEdges from node 0:")
for neighbor in G.successors(0):
    edge_data = G.get_edge_data(0, neighbor)
    for key, data in edge_data.items():
        print(f"  0 -> {neighbor} (relation: {data['relation']})")

# Example: Check for parallel edges
print(f"\nParallel edges between 0 and 1:")
parallel = G.get_edge_data(0, 1)
if parallel:
    print(f"  Found {len(parallel)} edges: {parallel}")

In [ ]:
source = 0
target = 2
nx.shortest_path_length(G, source=source, target=target)

In [ ]:
valid_paths = list(nx.all_shortest_paths(G, source=source, target=target))


In [ ]:
def find_shortest_paths_with_relation_sequences(
    G: nx.MultiDiGraph,
    source: int,
    target: int,
    allowed_relations: list[int] | None = None
) -> list[tuple[list[int], list[int]]]:
    """
    Find shortest paths and return both node sequences and relation sequences.
    
    Args:
        G: MultiDiGraph
        source: Source node
        target: Target node
        allowed_relations: Optional list of allowed relation IDs
        
    Returns:
        List of (node_path, relation_path) tuples
        Example: ([0, 1, 2], [1, 3]) means 0 --rel1--> 1 --rel3--> 2
    """
    # Get all shortest paths (nodes only)
    try:
        node_paths = list(nx.all_shortest_paths(G, source=source, target=target))
    except nx.NetworkXNoPath:
        return []
    
    result = []
    
    for node_path in node_paths:
        # For each path, get all possible relation sequences
        relation_sequences = get_relation_sequences_for_path(G, node_path, allowed_relations)
        
        for rel_seq in relation_sequences:
            result.append((node_path, rel_seq))
    
    return result

def get_relation_sequences_for_path(
    G: nx.MultiDiGraph,
    node_path: list[int],
    allowed_relations: list[int] | None = None
) -> list[list[int]]:
    """
    Get all possible relation sequences for a given node path.
    Handles parallel edges (multiple relations between same nodes).
    """
    if len(node_path) < 2:
        return [[]]
    
    # Get relations for each hop
    relations_per_hop = []
    
    for i in range(len(node_path) - 1):
        u, v = node_path[i], node_path[i + 1]
        edge_data = G.get_edge_data(u, v)
        
        if edge_data is None:
            return []  # No edge exists
        
        # Get all relations for this edge
        hop_relations = [
            data['relation'] 
            for key, data in edge_data.items()
        ]
        
        # Filter by allowed relations
        if allowed_relations is not None:
            hop_relations = [r for r in hop_relations if r in allowed_relations]
        
        if not hop_relations:
            return []  # No valid relations for this hop
        
        relations_per_hop.append(hop_relations)
    
    # Generate all combinations of relations
    import itertools
    relation_sequences = list(itertools.product(*relations_per_hop))
    
    return [list(seq) for seq in relation_sequences]

In [ ]:
paths = find_shortest_paths_with_relation_sequences(G, source, target)
for path in paths:
    print(path)

# Sufficient vs Necessary

In [ ]:
from dotenv import load_dotenv
from kge.common.actions.gnnkge_loader_action import GNNKGELoader

import torch

from kge.common.entities import  KGPredictionSubgraph
import os

In [ ]:
load_dotenv()


In [ ]:
import pandas as pd
file_name = "../explanations.tsv"
df = pd.read_csv(file_name, sep="\t")
df.head()

In [ ]:
idx = df["score_sufficient"].argmax()
df.loc[idx].explanation

# Separator

In [ ]:
import pandas as pd

In [ ]:
file_name = "../explanations.tsv"
df = pd.read_csv(file_name, sep="\t")
df.head()

In [ ]:


print(len(df))
for row in df.iloc[:3].itertuples():
    pathway = eval(row.pathway)
    score = row.score
    for s, o in pathway:
        print(f"{s} | {o}")
    print(f"----- {score*100:.3f}")

In [ ]:
top_per_entity = df.loc[df.groupby('pathway')['score'].idxmax()].sort_values('score', ascending=False)

remaining_rows = df[~df.index.isin(top_per_entity.index)].sort_values('score', ascending=False)

rerank_df = pd.concat([top_per_entity, remaining_rows], ignore_index=True)

print(len(rerank_df))
for row in rerank_df.iloc[:3].itertuples():
    pathway = eval(row.pathway)
    score = row.score
    for s, o in pathway:
        print(f"{s} | {o}")
    print(f"----- {score*100:.3f}")

In [ ]:
rerank_df.iloc[0].explanation

In [ ]:
df.sort_values(["score", "explanation"], ascending=False).head()

In [ ]:
rerank_df.sort_values(['score', "explanation"], ascending=False).head()

In [ ]:
import pandas as pd

# Sample data
data = {
    'score': [95, 87, 99, 78, 88, 91, 85, 90, 82, 89, 93, 86, 84, 96, 83, 94, 81, 97, 80, 98],
    'entities': ['A', 'B', 'A', 'C', 'B', 'D', 'C', 'A', 'E', 'D', 'F', 'B', 'G', 'E', 'H', 'F', 'I', 'G', 'J', 'I']
}
df = pd.DataFrame(data)
df

In [ ]:
# Get the top score per entity
top_per_entity = df.loc[df.groupby('entities')['score'].idxmax()].sort_values('score', ascending=False)

# Get all remaining rows (not the top ones per entity)
remaining_rows = df[~df.index.isin(top_per_entity.index)].sort_values('score', ascending=False)

# Combine them
pd.concat([top_per_entity, remaining_rows], ignore_index=True)

In [ ]:
df.loc[df.groupby('entities')['score'].idxmax()].sort_values('score', ascending=False)

# Separator

In [ ]:
print(os.getenv("DATA_VERSION"))
print(os.getenv("GNN_VERSION"))

In [ ]:
device = "cuda"

In [ ]:

project_dir = "/home/pablo.sanchez2/Documents/GitHub/project_spaice_ds/packages/pip/kge"

experiment_folder = os.path.join(project_dir, "outputs/prod/xkge-v1.1.0/")
experiment_data = GNNKGELoader.load_experiment(
        experiment_folder=experiment_folder,
        config_path = ".hydra/config.yaml",
        model_ckpt_path = "seed_0/model_checkpoint/last.ckpt",
        model_ckpt_is_lightning = True,
        device = device)

In [ ]:
model = experiment_data.model

model

In [ ]:
from datasets.common.constants import DataSplits
train_facts = experiment_data.data_processor.get_split(DataSplits.TRAIN).to(device)

In [ ]:
from torch_geometric.data import Batch


data_list = []
for i in range(10):
    triple_to_probe = torch.tensor([[i, 1 % 4, i+1]], dtype=torch.long, device=device)
    context_facts =  torch.tensor([[i+ 2, 1 % 4, i+1], [i+ 3, 1 % 4, i]], dtype=torch.long, device=device)
    pred_subgraph = KGPredictionSubgraph.from_facts(target_facts=triple_to_probe, 
                                                    context_facts=context_facts)
    
    data_list.append(pred_subgraph)


batch = Batch.from_data_list(data_list)
scores = model.score(batch) # shape [10, 1]



In [ ]:
scores.shape

In [ ]:
triple_to_probe = train_facts[[1199]]
print(triple_to_probe)

cond = train_facts[:, 2] == triple_to_probe[0, 2]
cond = torch.all(train_facts == triple_to_probe, dim=1)
context_facts = train_facts[~cond]
print(context_facts.shape)

In [ ]:

# context_facts = torch.tensor([[10, 2, 1040]], device=device)
pred_subgraph = KGPredictionSubgraph.from_facts(target_facts=triple_to_probe, context_facts=context_facts[:100])

pred_subgraph

In [ ]:
model.score(pred_subgraph)

In [ ]:
print(pred_subgraph.node_ids)
print(pred_subgraph.edge_index)
print(pred_subgraph.edge_type)
print(pred_subgraph.edge_label_index)
print(pred_subgraph.edge_label)